<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.2/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.2. Параметры, диалог и промпт как часть кода

**Модуль 2 · Урок 2 · 110 минут**

В уроке 2.1 вы собрали каркас и научились обращаться к модели. Сегодня разберётесь с тем, **что именно** вы ей отправляете: как хранить промпт, как управлять поведением модели параметрами и что происходит со стоимостью, когда разговор становится длинным.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Нужен ключ |
|---|---|---|---|
| 0 | Восстановим каркас урока 2.1 с нуля | 10 мин | нет |
| 1 | Напишем модуль промптов с параметрами и версиями | 20 мин | нет |
| 2 | Прогоним шесть самопроверок шаблонов | 10 мин | нет |
| 3 | Разберём параметры генерации и их поддержку у поставщиков | 15 мин | нет |
| 4 | Напишем модуль диалога | 20 мин | нет |
| 5 | Обрежем историю по бюджету | 10 мин | нет |
| 6 | Измерим, как история разгоняет стоимость | 15 мин | нет |
| 7 | Посчитаем стоимость на 1000 запросов — для итогового проекта | 10 мин | нет |
| 8 | Сравним ответы при разной температуре | 10 мин | да |

> **Ключ нужен только на последнем шаге.** Всё остальное считается и проверяется без обращений к API.

> **Если вы начинаете с чистого сеанса Colab**, файлы урока 2.1 уже стёрты — шаг 0 создаёт их заново. Это не дублирование, а проверка воспроизводимости: рабочий проект обязан собираться с нуля.

---
## Шаг 0. Восстанавливаем каркас урока 2.1

Colab стирает файлы, когда сеанс завершается, поэтому урок начинается со сборки проекта заново. Три ячейки: зависимости и структура, затем два модуля из прошлого урока — `config.py` и `client.py` **без единого изменения**.

Это не лишняя работа. Проект, который не собирается с чистого листа, сломан — просто вы обычно узнаёте об этом позже и хуже.

*Статус ячейки: проверено запуском.*

In [ ]:
REQUIREMENTS = [
    "openai==2.51.0",        # клиент к OpenAI-совместимым API
    "python-dotenv==1.2.2",  # чтение .env
]

import importlib.util, subprocess, sys
from pathlib import Path

def ensure(spec):
    name = spec.split("==")[0].replace("-", "_")
    if importlib.util.find_spec(name) is None:
        print(f"  устанавливаю {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
    else:
        print(f"  {name}: уже установлен")

print("Зависимости:")
for spec in REQUIREMENTS:
    ensure(spec)

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

print("\nПроект:", ROOT.resolve())

In [ ]:
%%writefile llm-project/llmcourse/config.py
"""Единая точка настройки. Урок 2.1.

Ключи НИКОГДА не пишутся в коде. Порядок поиска:
  1. Colab Secrets  (значок ключа слева в Colab)
  2. переменные окружения
  3. файл .env рядом с проектом
Если ключа нет — включается автономный режим на заглушке.
"""
import os
from pathlib import Path

ENV_KEYS = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")


def _from_colab(name):
    try:
        from google.colab import userdata          # есть только в Colab
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name, path=".env"):
    p = Path(path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        if k.strip() == name:
            return v.strip().strip('"').strip("'")
    return None


def get(name, default=None):
    """Достаёт значение из Colab Secrets, окружения или .env."""
    return _from_colab(name) or os.environ.get(name) or _from_dotenv(name) or default


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def settings():
    """Возвращает конфигурацию и признак автономного режима."""
    cfg = {k: get(k) for k in ENV_KEYS}
    cfg["OFFLINE"] = not bool(cfg["LLM_API_KEY"])
    cfg["MODEL"] = cfg["LLM_MODEL"] or "demo-model"
    return cfg


def describe():
    """Человекочитаемый отчёт об окружении."""
    cfg = settings()
    where = "Google Colab" if in_colab() else "локальная среда"
    return "\n".join([
        f"Среда:            {where}",
        f"Модель:           {cfg['MODEL']}",
        f"Базовый адрес:    {cfg['LLM_BASE_URL'] or 'не задан'}",
        f"Ключ:             {'найден' if not cfg['OFFLINE'] else 'НЕ найден'}",
        f"Режим:            {'автономный (заглушка)' if cfg['OFFLINE'] else 'обращение к API'}",
    ])


In [ ]:
%%writefile llm-project/llmcourse/client.py
"""Клиент для работы с языковой моделью. Уроки 2.1–2.2.

Написан на OpenAI-совместимый интерфейс: работает с российскими API
и с локальными рантаймами. Смена поставщика — правка .env, не кода.
Без ключа работает в автономном режиме на заглушке.
"""
import time, random, hashlib
from dataclasses import dataclass
from . import config


@dataclass
class Usage:
    """Накопительный счётчик расхода."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    price_in: float = 0.0     # рублей за 1000 входных токенов
    price_out: float = 0.0    # рублей за 1000 выходных

    @property
    def cost(self):
        return (self.tokens_in / 1000 * self.price_in +
                self.tokens_out / 1000 * self.price_out)

    def report(self):
        return (f"обращений: {self.calls}   "
                f"токенов: {self.tokens_in} вход / {self.tokens_out} выход   "
                f"стоимость: {self.cost:.4f} руб.")


def approx_tokens(text):
    """Грубая ОЦЕНКА числа токенов до вызова API.

    Это оценка, а не замер: точное число даёт токенизатор конкретной
    модели (см. урок 1.2). Нужна, чтобы прикинуть стоимость заранее.
    """
    return max(1, len(text) // 3)


class LLM:
    def __init__(self, price_in=0.0, price_out=0.0, max_retries=4, timeout=60):
        cfg = config.settings()
        self.offline = cfg["OFFLINE"]
        self.model = cfg["MODEL"]
        self.base_url = cfg["LLM_BASE_URL"]
        self.max_retries = max_retries
        self.usage = Usage(price_in=price_in, price_out=price_out)
        self._client = None
        if not self.offline:
            from openai import OpenAI
            self._client = OpenAI(base_url=self.base_url,
                                  api_key=cfg["LLM_API_KEY"],
                                  timeout=timeout)

    def _offline_answer(self, messages):
        """Детерминированный ответ: одинаковый запрос — одинаковый ответ."""
        text = " ".join(m["content"] for m in messages)
        h = hashlib.sha256(text.encode()).hexdigest()[:6]
        return (f"[автономный режим] Ответ-заглушка {h}. "
                f"Получено сообщений: {len(messages)}, символов: {len(text)}. "
                f"Подставьте ключ, чтобы обратиться к модели.")

    @staticmethod
    def _is_retryable(e):
        """Повторяем только то, что имеет шанс пройти со второго раза."""
        if type(e).__name__ in ("RateLimitError", "APITimeoutError",
                                "APIConnectionError", "InternalServerError",
                                "TimeoutError", "ConnectionError"):
            return True
        return getattr(e, "status_code", None) in (408, 429, 500, 502, 503, 504)

    def _with_retry(self, fn):
        """Экспоненциальная задержка со случайной добавкой."""
        for attempt in range(self.max_retries):
            try:
                return fn()
            except Exception as e:
                if not self._is_retryable(e) or attempt == self.max_retries - 1:
                    raise
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.3))

    def ask(self, prompt, system=None, temperature=0.2, max_tokens=None):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        self.usage.calls += 1

        if self.offline:
            answer = self._offline_answer(messages)
            self.usage.tokens_in += approx_tokens(" ".join(m["content"] for m in messages))
            self.usage.tokens_out += approx_tokens(answer)
            return answer

        def call():
            kw = dict(model=self.model, messages=messages, temperature=temperature)
            if max_tokens:
                kw["max_tokens"] = max_tokens
            return self._client.chat.completions.create(**kw)

        resp = self._with_retry(call)
        u = getattr(resp, "usage", None)
        if u:
            self.usage.tokens_in += getattr(u, "prompt_tokens", 0)
            self.usage.tokens_out += getattr(u, "completion_tokens", 0)
        return resp.choices[0].message.content


In [ ]:
import importlib
importlib.invalidate_caches()
from llmcourse import config, client
importlib.reload(config); importlib.reload(client)

print(config.describe())
print()
print("Каркас урока 2.1 на месте. Работаем дальше.")

---
## Шаг 1. Промпт — не строка, а объект с версией

Пока промпт живёт внутри функции, с ним нельзя работать: не видно, какие в нём параметры, нельзя понять, какой версией получен вчерашний ответ, и правка формулировки требует лезть в код вызова.

Выносим промпт в отдельный модуль. У него появляются:

| Свойство | Зачем |
|---|---|
| `name` | по имени берут промпт из реестра |
| `version` | видно в журнале, чем получен ответ |
| `system` | правила работы модели |
| `template` | сама задача, с параметрами в фигурных скобках |

**Почему не `str.format`.** Во-первых, `.format` умеет обращаться к атрибутам объекта — запись `{x.__class__}` в шаблоне сработает. Если шаблон пришёл извне, это дыра в безопасности. Во-вторых, нам нужна подстановка и ничего сверх того: всё лишнее в инструменте рано или поздно кто-нибудь применит.

Правила шаблона мы намеренно взяли те же, что в Python, — чтобы знание переносилось:

```
{name}   параметр
{{       литеральная открывающая скобка
}}       литеральная закрывающая скобка
```

Удвоение скобок понадобится сразу же: как только вы попросите модель вернуть JSON, в шаблоне появятся литеральные скобки.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/prompts.py
"""Промпт как часть программы. Урок 2.2.

Промпт хранится отдельно от кода вызова: у него есть имя, версия и явный
список параметров. Это позволяет менять формулировку, не трогая логику,
и видеть в журнале, какой версией промпта получен ответ.

Почему не str.format. Во-первых, .format умеет обращаться к атрибутам объекта
("{x.__class__}"), и на шаблоне, пришедшем извне, это дыра в безопасности.
Во-вторых, нам нужна подстановка и ничего больше — а всё лишнее в инструменте
рано или поздно кто-нибудь применит.

Правила шаблона те же, что в Python, чтобы знание переносилось:
    {name}  — параметр
    {{      — литеральная открывающая скобка
    }}      — литеральная закрывающая скобка
Одиночная } без пары — ошибка. Это не придирка: чаще всего она означает
незакрытый или неверно записанный параметр.
"""
import re
from dataclasses import dataclass

# Имя параметра — любой допустимый идентификатор, включая кириллицу: Python это
# разрешает. Но в примерах курса имена латинские — так принято, и так их видно
# в чужом коде без сюрпризов с раскладкой.
_NAME = re.compile(r"[^\W\d]\w*")


class PromptError(ValueError):
    """Ошибка в шаблоне промпта или в его заполнении."""


def parse(text):
    """Разбирает шаблон в список кусков: ("lit", строка) или ("field", имя)."""
    out, buf, i, n = [], [], 0, len(text)

    def flush():
        if buf:
            out.append(("lit", "".join(buf)))
            buf.clear()

    while i < n:
        ch = text[i]
        if ch == "{":
            if i + 1 < n and text[i + 1] == "{":
                buf.append("{"); i += 2; continue
            j = text.find("}", i + 1)
            if j == -1:
                raise PromptError("незакрытая « { » в шаблоне")
            name = text[i + 1:j]
            if not _NAME.fullmatch(name):
                raise PromptError(
                    f"недопустимое имя параметра: {{{name}}}. "
                    "Для литеральной скобки используйте {{ и }}")
            flush()
            out.append(("field", name))
            i = j + 1
            continue
        if ch == "}":
            if i + 1 < n and text[i + 1] == "}":
                buf.append("}"); i += 2; continue
            raise PromptError(
                "одиночная « } » в шаблоне. Для литеральной скобки пишите }}")
        buf.append(ch); i += 1
    flush()
    return out


@dataclass(frozen=True)
class Prompt:
    name: str
    version: str
    template: str
    system: str = ""

    @property
    def fields(self):
        """Имена параметров, которые нужно передать при заполнении."""
        parts = parse(self.template) + parse(self.system)
        return sorted({v for kind, v in parts if kind == "field"})

    def render(self, **values):
        """Заполняет шаблон. Молча ничего не проглатывает."""
        need, got = set(self.fields), set(values)
        if need - got:
            raise PromptError(
                f"{self.label()}: не переданы параметры {sorted(need - got)}")
        if got - need:
            raise PromptError(
                f"{self.label()}: лишние параметры {sorted(got - need)}. "
                f"Ожидались {self.fields}. Опечатка в имени — частая причина")

        def fill(text):
            return "".join(
                v if kind == "lit" else str(values[v]) for kind, v in parse(text))

        return {"system": fill(self.system), "user": fill(self.template)}

    def label(self):
        """Метка для журнала: по ней потом видно, чем получен ответ."""
        return f"{self.name}@{self.version}"


class Registry:
    """Все промпты проекта в одном месте."""

    def __init__(self):
        self._items = {}

    def add(self, prompt):
        old = self._items.get(prompt.name)
        if old is not None and old.version == prompt.version:
            raise PromptError(
                f"промпт {prompt.name} версии {prompt.version} уже зарегистрирован. "
                "Меняете формулировку — поднимите версию")
        self._items[prompt.name] = prompt
        return prompt

    def get(self, name):
        if name not in self._items:
            raise PromptError(f"промпт {name} не найден. Есть: {sorted(self._items)}")
        return self._items[name]

    def names(self):
        return sorted(self._items)


---
## Шаг 2. Самопроверки шаблонов

Шесть проверок. Обратите внимание на четвёртую и пятую — они про то, ради чего писался разбор шаблона вручную.

*Статус ячейки: проверено запуском.*

In [ ]:
import importlib
importlib.invalidate_caches()        # файл создан после того, как Python осмотрел папку
import llmcourse.prompts
importlib.reload(llmcourse.prompts)
from llmcourse.prompts import Prompt, Registry, PromptError

extract = Prompt(
    name="extract",
    version="1.0",
    system="Ты аккуратный аналитик. Отвечай только фактами из текста.",
    template="Найди в тексте {target}.\n\nТекст:\n{text}",
)

assert extract.fields == ["target", "text"]
print("[ok] параметры шаблона распознаны:", extract.fields)

r = extract.render(target="все даты", text="Совещание перенесено на 5 мая.")
assert r["user"].startswith("Найди в тексте все даты")
print("[ok] заполнение работает")

try:
    extract.render(target="даты")
    raise SystemExit("проверка провалена")
except PromptError as e:
    print("[ok] пропущенный параметр:", e)

try:
    extract.render(target="д", text="т", tekst="опечатка")
    raise SystemExit("проверка провалена")
except PromptError as e:
    print("[ok] лишний параметр:", str(e)[:70], "...")

TPL = 'Верни {{"key": {value}}}'
j = Prompt(name="json", version="1.0", template=TPL)
assert j.fields == ["value"]
assert j.render(value=5)["user"] == TPL.format(value=5)
print("[ok] JSON-скобки в шаблоне:", j.render(value=5)["user"])

try:
    Prompt(name="x", version="1.0", template="{p.__class__}").fields
    raise SystemExit("проверка провалена")
except PromptError:
    print("[ok] обращение к атрибутам запрещено — в отличие от str.format")

reg = Registry()
reg.add(extract)
try:
    reg.add(Prompt(name="extract", version="1.0", template="другой текст"))
    raise SystemExit("проверка провалена")
except PromptError as e:
    print("[ok] дубль версии отвергнут:", str(e)[:60], "...")

reg.add(Prompt(name="extract", version="1.1",
               system=extract.system,
               template="Выпиши {target} из текста ниже. Только то, что есть в тексте.\n\n{text}"))
print("[ok] новая версия принята, метка для журнала:", reg.get("extract").label())

print("\nМодуль промптов работает.")

---
## Шаг 3. Параметры генерации

Четыре параметра, которыми пользуются постоянно.

| Параметр | Что делает | Когда трогать |
|---|---|---|
| `temperature` | степень случайности ответа | ниже — для извлечения данных и классификации, выше — для генерации вариантов |
| `top_p` | доля вероятностной массы, из которой выбираются токены | альтернатива температуре; вместе обычно не крутят |
| `max_tokens` | потолок длины ответа | всегда: защищает от неожиданно длинного и дорогого ответа |
| `stream` | выдавать ответ по частям | когда ответ читает человек и важно не ждать молча |

**Главная ошибка** — крутить `temperature` и `top_p` одновременно. Оба управляют одним и тем же — шириной выбора, — и их совместное действие предсказать трудно. Меняйте что-то одно.

### Чего нет у всех

Совместимость с интерфейсом OpenAI частичная, и наборы параметров различаются. Ниже — то, что удалось подтвердить по официальным документам на 30.07.2026. Таблица не полна: у вашего поставщика может быть иначе.

| Параметр | GigaChat | Ollama |
|---|---|---|
| `temperature`, `top_p`, `max_tokens`, `stream` | есть | есть |
| `seed` — повторяемость ответа | **нет в спецификации** | есть |
| `stop` — стоп-строки | **нет в спецификации** | есть |
| `presence_penalty`, `frequency_penalty` | **нет в спецификации** | есть |
| `n` — несколько вариантов за запрос | **нет в спецификации** | **не поддерживается** |
| `logit_bias` | **нет в спецификации** | **не поддерживается** |
| `repetition_penalty` | есть (нет у OpenAI) | — |
| `update_interval` — темп потоковой выдачи | есть (нет у OpenAI) | — |

Источники: OpenAPI-спецификация GigaChat (схема `Chat`) и раздел «OpenAI compatibility» документации Ollama. Ссылки — в материалах урока.

### Ограничение, на котором спотыкаются

В GigaChat системное сообщение должно быть **ровно одно и первым** в массиве. Иначе API отвечает ошибкой 422 с текстом `Invalid params: system message must be the first message`. Это зафиксировано в спецификации, и наш модуль диалога соблюдает правило по умолчанию — чтобы код оставался переносимым.

*Статус ячейки: сведения из документации, проверены переходом по ссылкам; поведение конкретного поставщика требует проверки на живом ключе.*

In [ ]:
# Готовим параметры запроса и сразу проверяем себя на типичной ошибке.

def build_params(temperature=None, top_p=None, max_tokens=None, stream=False):
    """Собирает параметры запроса, отсекая заведомо неудачные сочетания."""
    if temperature is not None and top_p is not None:
        raise ValueError(
            "temperature и top_p управляют одним и тем же. "
            "Задайте что-то одно, иначе результат трудно объяснить")
    if temperature is not None and not 0 < temperature <= 2:
        raise ValueError("temperature: разумный диапазон 0 < t <= 2")
    if top_p is not None and not 0 <= top_p <= 1:
        raise ValueError("top_p: диапазон от 0 до 1")
    if max_tokens is not None and max_tokens <= 0:
        raise ValueError("max_tokens должен быть больше нуля")

    params = {}
    if temperature is not None:
        params["temperature"] = temperature
    if top_p is not None:
        params["top_p"] = top_p
    if max_tokens is not None:
        params["max_tokens"] = max_tokens
    if stream:
        params["stream"] = True
    return params


print(build_params(temperature=0.2, max_tokens=300))

for bad in (dict(temperature=0.2, top_p=0.9), dict(temperature=5), dict(max_tokens=0)):
    try:
        build_params(**bad)
        raise SystemExit("проверка провалена: " + str(bad))
    except ValueError as e:
        print("[ok]", str(e)[:72])

print("\nПараметры собираются, неудачные сочетания отсекаются.")

---
## Шаг 4. Диалог: модель ничего не помнит

Это место, где ломается интуиция. Когда вы переписываетесь с моделью в чате, кажется, что она помнит разговор. Не помнит. Каждый запрос обрабатывается с нуля, а «память» — это вся предыдущая переписка, которую интерфейс молча прикладывает к каждому вашему сообщению.

Отсюда два следствия:

1. **Стоимость каждого следующего хода растёт**, потому что вы заново отправляете всё сказанное раньше.
2. **История упрётся в размер контекста** модели, и тогда запрос перестанет проходить.

Значит, историю надо обрезать — осознанно, а не когда всё сломается.

Что мы закладываем в модуль:

- системное сообщение не выбрасывается никогда: в нём правила работы, без него модель меняет поведение целиком;
- последний ход пользователя сохраняется всегда, иначе запрос теряет смысл;
- отбрасываются самые старые ходы;
- если системное сообщение само не влезает в бюджет — внятная ошибка, а не тихая порча запроса.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/dialog.py
"""Многошаговый диалог. Урок 2.2.

Модель не помнит предыдущие сообщения. Память диалога — это то, что вы
сами прикладываете к каждому запросу. Отсюда два следствия, которые
и составляют содержание модуля:

1. История растёт, и вместе с ней растёт стоимость каждого следующего хода.
2. История рано или поздно упрётся в размер контекста модели.

Поэтому историю приходится обрезать, и делать это осознанно.

Ограничение поставщика, которое надо учитывать. В GigaChat системный промпт
должен быть ровно один и первым сообщением; иначе API отвечает ошибкой 422
«system message must be the first message» (см. OpenAPI-спецификацию, поле
messages). Наш класс держит это правило по умолчанию — так код остаётся
переносимым.
"""
from dataclasses import dataclass, field


def approx_tokens(text):
    """ОЦЕНКА числа токенов, не замер. См. урок 2.1."""
    return max(1, len(text) // 3)


@dataclass
class Dialog:
    system: str = ""
    budget_tokens: int = 4000          # сколько токенов истории готовы отправлять
    _turns: list = field(default_factory=list)   # [(role, content), ...] без system

    # ── наполнение ────────────────────────────────────────────────────
    def add(self, role, content):
        if role == "system":
            raise ValueError(
                "системное сообщение задаётся один раз при создании диалога: "
                "многие поставщики принимают ровно один system и только первым")
        if role not in ("user", "assistant"):
            raise ValueError(f"неизвестная роль: {role}")
        self._turns.append((role, content))
        return self

    def user(self, content):
        return self.add("user", content)

    def assistant(self, content):
        return self.add("assistant", content)

    # ── подсчёт ───────────────────────────────────────────────────────
    def tokens(self, turns=None):
        turns = self._turns if turns is None else turns
        total = approx_tokens(self.system) if self.system else 0
        return total + sum(approx_tokens(c) for _, c in turns)

    # ── обрезка ───────────────────────────────────────────────────────
    def fit(self):
        """Возвращает историю, укладывающуюся в бюджет.

        Системное сообщение не выбрасывается никогда: в нём правила работы,
        без него модель меняет поведение целиком. Обрезаются самые старые
        ходы. Последний ход пользователя сохраняется всегда — иначе запрос
        теряет смысл.
        """
        base = approx_tokens(self.system) if self.system else 0
        if base >= self.budget_tokens and self._turns:
            raise ValueError(
                "системное сообщение само не помещается в бюджет: "
                f"{base} токенов при бюджете {self.budget_tokens}")

        kept, total = [], base
        for role, content in reversed(self._turns):
            need = approx_tokens(content)
            if total + need > self.budget_tokens and kept:
                break
            total += need
            kept.append((role, content))
        return list(reversed(kept))

    def messages(self):
        """Готовый массив messages для запроса к модели."""
        out = [{"role": "system", "content": self.system}] if self.system else []
        out += [{"role": r, "content": c} for r, c in self.fit()]
        return out

    def dropped(self):
        """Сколько старых ходов не вошло в запрос."""
        return len(self._turns) - len(self.fit())


In [ ]:
import importlib
importlib.invalidate_caches()
import llmcourse.dialog
importlib.reload(llmcourse.dialog)
from llmcourse.dialog import Dialog

d = Dialog(system="Ты помощник справочной службы. Отвечай кратко.", budget_tokens=100)
d.user("Здравствуйте").assistant("Здравствуйте! Чем помочь?").user("Как поменять тариф?")

for m in d.messages():
    print(f"{m['role']:>9} | {m['content'][:52]}")

assert d.messages()[0]["role"] == "system"
print("\n[ok] системное сообщение первое")

try:
    d.add("system", "ещё одно системное")
    raise SystemExit("проверка провалена")
except ValueError as e:
    print("[ok]", str(e)[:78])

print("\nДиалог собирается корректно.")

---
## Шаг 5. Обрезка истории

Смотрим, что происходит с длинным разговором при ограниченном бюджете.

*Статус ячейки: проверено запуском.*

In [ ]:
long = Dialog(system="Ты помощник справочной службы. Отвечай кратко.", budget_tokens=200)
for i in range(1, 21):
    long.user(f"Вопрос номер {i}: расскажите, пожалуйста, подробнее об условиях.")
    long.assistant(f"Ответ номер {i}: условия описаны в разделе {i} регламента.")

print("Всего ходов в истории:", len(long._turns))
print("Влезло в бюджет:      ", len(long.fit()))
print("Отброшено старых:     ", long.dropped())
print("Оценка объёма запроса:", long.tokens(long.fit()), "токенов при бюджете", long.budget_tokens)

kept = long.fit()
assert kept[-1][1].startswith("Ответ номер 20"), kept[-1]
print("\n[ok] самый свежий ход сохранён")
assert long.messages()[0]["role"] == "system"
print("[ok] системное сообщение на месте")

huge = Dialog(system="П" * 3000, budget_tokens=100)
huge.user("вопрос")
try:
    huge.fit()
    raise SystemExit("проверка провалена")
except ValueError as e:
    print("[ok]", str(e)[:88])

print("\nОбрезка работает предсказуемо.")

---
## Шаг 6. Во что обходится длинный разговор

Теперь измерим то, о чём говорили на шаге 4. Каждый ход заново отправляет всю переписку — значит, расход растёт быстрее, чем число ходов.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/costs.py
"""Расчёт стоимости эксплуатации. Урок 2.2.

Нужен для итогового проекта модуля: там требуется расчёт стоимости
на 1000 запросов. Все цены задаются вызывающей стороной — модуль не знает
и не выдумывает тарифов.

Две схемы оплаты, встречающиеся у поставщиков:
  раздельная — своя цена за входные и за выходные токены;
  пакетная   — единая цена за токен (тогда price_in == price_out).
"""
from dataclasses import dataclass


@dataclass(frozen=True)
class Price:
    """Цена в рублях за 1000 токенов."""
    per_1k_in: float
    per_1k_out: float

    @classmethod
    def flat(cls, per_1k):
        """Единая ставка: пакетная схема оплаты."""
        return cls(per_1k, per_1k)


def one_call(tokens_in, tokens_out, price):
    return tokens_in / 1000 * price.per_1k_in + tokens_out / 1000 * price.per_1k_out


def run(n_requests, tokens_in, tokens_out, price, retry_factor=1.0):
    """Стоимость серии одинаковых запросов.

    retry_factor — во сколько раз больше обращений реально уходит из-за
    повторов. 1.0 означает «повторов нет». Значение берётся из наблюдений,
    а не из головы: посчитайте долю повторов на пилотном прогоне.
    """
    if retry_factor < 1:
        raise ValueError("retry_factor не может быть меньше 1")
    calls = n_requests * retry_factor
    return {
        "calls": calls,
        "tokens_in": calls * tokens_in,
        "tokens_out": calls * tokens_out,
        "cost": calls * one_call(tokens_in, tokens_out, price),
    }


def conversation(n_turns, user_tokens, answer_tokens, price, system_tokens=0):
    """Стоимость диалога из n_turns ходов БЕЗ обрезки истории.

    На каждом ходу заново отправляется вся предыдущая переписка. Поэтому
    суммарный расход растёт не линейно, а квадратично по числу ходов.
    Возвращает список по ходам — чтобы это было видно, а не постулировалось.
    """
    rows, history, total = [], system_tokens, 0.0
    for turn in range(1, n_turns + 1):
        history += user_tokens
        c = one_call(history, answer_tokens, price)
        total += c
        history += answer_tokens
        rows.append({"turn": turn, "tokens_in": history - answer_tokens,
                     "cost": c, "total": total})
    return rows


In [ ]:
import importlib
importlib.invalidate_caches()
import llmcourse.costs
importlib.reload(llmcourse.costs)
from llmcourse.costs import Price, one_call, run, conversation

# Ставка условная — подставьте тариф своего поставщика.
PRICE = Price.flat(0.065)     # руб. за 1000 токенов, единая ставка

rows = conversation(n_turns=12, user_tokens=120, answer_tokens=250,
                    price=PRICE, system_tokens=60)

print(f"{'ход':>4} | {'отправлено токенов':>19} | {'стоимость хода':>15} | {'накопительно':>13}")
print("-" * 62)
for r in rows:
    print(f"{r['turn']:>4} | {r['tokens_in']:>19} | {r['cost']:>15.4f} | {r['total']:>13.4f}")

first, last = rows[0]["cost"], rows[-1]["cost"]
linear = first * len(rows)
print()
print(f"Двенадцатый ход дороже первого в {last / first:.1f} раза")
print(f"Фактически потрачено:            {rows[-1]['total']:.3f} руб.")
print(f"Если бы рост был линейным:       {linear:.3f} руб.")
print(f"Переплата из-за истории:         {rows[-1]['total'] / linear:.1f}x")

assert rows[-1]["total"] > linear
print("\n[ok] рост расхода подтверждён замером, а не постулирован")

**Что с этим делать.** Три приёма, по возрастанию сложности:

1. **Ограничить бюджет истории** — то, что мы сделали на шаге 5. Просто и предсказуемо.
2. **Пересказывать старую часть** — заменить первые десять ходов одним кратким изложением. Дешевле, но нужен лишний вызов модели и теряются детали.
3. **Не вести диалог там, где он не нужен.** Самый недооценённый приём: для извлечения данных, классификации и разметки история не нужна вовсе — каждый запрос независим. Диалог там просто дорогая привычка.

---
## Шаг 7. Расчёт на 1000 запросов

Этот расчёт понадобится в **итоговом проекте модуля** — там он входит в состав обязательных материалов. Сделайте его сейчас на своих числах и сохраните.

*Статус ячейки: проверено запуском.*

In [ ]:
# ── подставьте свои значения ──────────────────────────────────────
N_REQUESTS   = 1000     # сколько запросов в расчётном периоде
TOKENS_IN    = 1800     # средний объём запроса вместе с промптом
TOKENS_OUT   = 400      # средний объём ответа
RETRY_FACTOR = 1.10     # доля повторов: 1.10 = на 10% обращений больше
PRICE_1K     = 0.065    # руб. за 1000 токенов по вашему тарифу
# ──────────────────────────────────────────────────────────────────

price = Price.flat(PRICE_1K)
base = run(N_REQUESTS, TOKENS_IN, TOKENS_OUT, price)
real = run(N_REQUESTS, TOKENS_IN, TOKENS_OUT, price, retry_factor=RETRY_FACTOR)

print(f"Запросов:                 {N_REQUESTS}")
print(f"Токенов на запрос:        {TOKENS_IN} + {TOKENS_OUT}")
print(f"Обращений с учётом повторов: {real['calls']:.0f}")
print()
print(f"Без повторов:             {base['cost']:.2f} руб.")
print(f"С повторами:              {real['cost']:.2f} руб.")
print(f"На один запрос:           {real['cost'] / N_REQUESTS:.4f} руб.")
print()
print(f"Оценка на 10 000 запросов:  {real['cost'] * 10:.2f} руб.")
print(f"Оценка на 100 000 запросов: {real['cost'] * 100:.2f} руб.")

print("\nЭто ОЦЕНКА. Перед защитой проекта сверьте её с фактическим usage")
print("на пилотной выборке — расхождение покажет, насколько ей можно верить.")

---
## Шаг 8. Живая проверка: одна задача при разной температуре

Единственный шаг, которому нужен ключ. Берём один и тот же промпт и запускаем его дважды: при низкой температуре и при высокой.

Что смотреть: при низкой температуре ответы на повторных запусках должны быть похожи, при высокой — заметно расходиться. Если разницы нет вовсе — возможно, ваш поставщик игнорирует параметр; это как раз тот случай, ради которого на шаге 3 стоит оговорка о частичной совместимости.

*Статус ячейки: требует проверки на живом ключе. Автор материалов эту ячейку с настоящим ключом не запускал.*

In [ ]:
# Требуется ключ. Без него ячейка честно скажет об этом и не будет притворяться.
import importlib
importlib.invalidate_caches()
from llmcourse import config, client as llm_client
importlib.reload(config); importlib.reload(llm_client)

HAVE_KEY = not config.settings().get("OFFLINE", True)

if not HAVE_KEY:
    print()
    print("Ключ не подключён — шаг пропускается.")
    print("Это не ошибка: шаги 0-7 дают всё содержание урока.")
    print("Вернитесь сюда, когда получите доступ.")
else:
    task = Prompt(
        name="summary", version="1.0",
        system="Ты редактор. Пиши по-русски, без вводных слов.",
        template="Одним предложением сформулируй главную мысль:\n\n{text}",
    )
    TEXT = ("Рабочее окружение состоит из четырёх слоёв, и диагностика неисправности "
            "начинается с верхнего: сначала выясняют, где именно запускается код.")

    llm = llm_client.LLM()
    for t in (0.1, 1.0):
        print(f"\n--- temperature = {t} ---")
        for attempt in (1, 2):
            r = task.render(text=TEXT)
            out = llm.ask(r["user"], system=r["system"], temperature=t, max_tokens=120)
            print(f"  прогон {attempt}: {out}")
    print("\nСравните два прогона внутри каждой температуры.")

---
## Задание

1. **Вторая версия промпта.** Возьмите промпт `extract` версии 1.0 и напишите версию 1.1, добавив в системное сообщение запрет придумывать то, чего нет в тексте. Зарегистрируйте обе в реестре и убедитесь, что метка в журнале различается.

2. **Свой расчёт.** Заполните шаг 7 числами своей задачи: объём запроса, объём ответа, тариф вашего поставщика. Сохраните результат — он войдёт в итоговый проект.

3. **Обрезка на своих данных.** Подберите `budget_tokens` так, чтобы в запрос помещалось примерно шесть последних ходов. Объясните письменно, почему выбрали именно это число.

### Повышенной сложности

4. Добавьте в `Dialog` метод, который вместо отбрасывания старых ходов заменяет их одной строкой-пересказом. Оцените, во сколько обходится сам пересказ и с какого числа ходов он окупается.

5. Напишите функцию, которая по журналу обращений считает фактическую долю повторов и возвращает `retry_factor` для расчёта. Сейчас вы задаёте его на глаз — а он должен браться из наблюдений.

---
## Чек-лист

- [ ] Модуль `prompts.py` создан, все шесть самопроверок печатают «ок»
- [ ] Понимаю, почему в шаблоне для JSON скобки удваиваются
- [ ] Модуль `dialog.py` создан, системное сообщение не теряется при обрезке
- [ ] Видел на своих числах, что двенадцатый ход дороже первого в несколько раз
- [ ] Расчёт на 1000 запросов заполнен моими значениями и сохранён
- [ ] Могу объяснить, почему `temperature` и `top_p` не крутят одновременно
- [ ] Знаю, где посмотреть, поддерживает ли мой поставщик нужный параметр

## Частые проблемы

| Симптом | Причина и что делать |
|---|---|
| `PromptError: не переданы параметры` | В шаблоне есть параметр, которого нет в вызове. Посмотрите `prompt.fields` — там точный список |
| `PromptError: лишние параметры` | Опечатка в имени. Модуль намеренно не проглатывает такое молча: иначе промпт уходит незаполненным |
| `PromptError: одиночная }` | В шаблоне литеральная скобка. Удвойте: `}}` |
| `ImportError: cannot import name` | Файл создан после того, как Python осмотрел папку. Перед импортом нужен `importlib.invalidate_caches()` |
| Ответ обрывается | Упёрлись в `max_tokens`. Проверьте `finish_reason` — при значении `length` поднимайте потолок |
| Ошибка 422 про system message | Системных сообщений больше одного или оно не первое. Наш `Dialog` это предотвращает |
| Температура ничего не меняет | Возможно, поставщик игнорирует параметр. Сверьтесь с его документацией |

## Что дальше

Урок 2.3 — **структурированный вывод**. Вы попросите модель вернуть не текст, а данные заданной формы, и научитесь поступать с ответом, который этой форме не соответствует. Промпты для этого уже есть — сегодня вы научились их хранить.